In [127]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier, plot_importance
import seaborn as sns
import matplotlib.pyplot as plt
import shap
# must be set so that figures do not get cut off
plt.rcParams['figure.autolayout'] = True

In [1]:
# import drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Define Folder Paths

In [137]:
# folder path to ML data
ML_data_folder_path = '/content/drive/My Drive/capstone_data/ml_data_automation_test/'

# ----------------- SEASONAL MODEL FOLDER PATHS ---------------------- #

# folder path to save classification reports
classification_report_folder_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/seasonal_models/classification_reports/'

# folder path to save confusion matrix figures
confusion_matrix_folder_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/seasonal_models/confusion_matrices/'

# folder path to store feature importance figures
feature_importance_folder_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/seasonal_models/feature_importance_plots/'

# probablistic forecast details path
forecast_details_folder_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/seasonal_models/probablistic_forecast_details/'

# folder path to save trained models
model_folder_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/seasonal_models/trained_models/'

# folder path to shap feature importance figures
shap_feature_importance_folder_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/seasonal_models/shap_feature_importance_plots/'

# --------------------------------------------------------------------------------- #

# ----------------- MONTHLY MODEL FOLDER PATHS ---------------------- #

classification_report_folder_monthly = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_models/classification_reports/'

confusion_matrix_folder_monthly = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_models/confusion_matrices/'

feature_importance_folder_monthly = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_models/feature_importance_plots/'

forecast_details_folder_monthly = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_models/probablistic_forecast_details/'

model_folder_monthly = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_models/trained_models/'

shap_feature_importance_folder_monthly = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_models/shap_feature_importance_plots/'

# ------------------------------------------------------------------------------- #

# ---------------------------- MONTHLY MODEL (SEASONAL TERCILE) FOLDER PATHS ----------------- #
classification_report_folder_monthly_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_seasonal_tercile_models/classification_reports/'

confusion_matrix_folder_monthly_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_seasonal_tercile_models/confusion_matrices/'

feature_importance_folder_monthly_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_seasonal_tercile_models/feature_importance_plots/'

forecast_details_folder_monthly_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_seasonal_tercile_models/probablistic_forecast_details/'

model_folder_monthly_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_seasonal_tercile_models/trained_models/'

shap_feature_importance_folder_monthly_seasonal = '/content/drive/My Drive/capstone_data/ML_results/XGBoost/monthly_seasonal_tercile_models/shap_feature_importance_plots/'
# ----------------------------------------------------------------------------------------------------------------------------------------------------------------

# normalize all folder paths
ML_data_folder_path = os.path.normpath(ML_data_folder_path)

classification_report_folder_seasonal = os.path.normpath(classification_report_folder_seasonal)
confusion_matrix_folder_seasonal = os.path.normpath(confusion_matrix_folder_seasonal)
feature_importance_folder_seasonal = os.path.normpath(feature_importance_folder_seasonal)
forecast_details_folder_seasonal = os.path.normpath(forecast_details_folder_seasonal)
model_folder_seasonal = os.path.normpath(model_folder_seasonal)
shap_feature_importance_folder_seasonal = os.path.normpath(shap_feature_importance_folder_seasonal)

classification_report_folder_monthly = os.path.normpath(classification_report_folder_monthly)
confusion_matrix_folder_monthly = os.path.normpath(confusion_matrix_folder_monthly)
feature_importance_folder_monthly = os.path.normpath(feature_importance_folder_monthly)
forecast_details_folder_monthly = os.path.normpath(forecast_details_folder_monthly)
model_folder_monthly = os.path.normpath(model_folder_monthly)
shap_feature_importance_folder_monthly = os.path.normpath(shap_feature_importance_folder_monthly)

classification_report_folder_monthly_seasonal = os.path.normpath(classification_report_folder_monthly_seasonal)
confusion_matrix_folder_monthly_seasonal = os.path.normpath(confusion_matrix_folder_monthly_seasonal)
feature_importance_folder_monthly_seasonal = os.path.normpath(feature_importance_folder_monthly_seasonal)
forecast_details_folder_monthly_seasonal = os.path.normpath(forecast_details_folder_monthly_seasonal)
model_folder_monthly_seasonal = os.path.normpath(model_folder_monthly_seasonal)
shap_feature_importance_folder_monthly_seasonal = os.path.normpath(shap_feature_importance_folder_monthly_seasonal)

# Current XGBoost Model:
- Evaluation Metric: Multi-Class Log Loss (mlogloss)
- No Cross Validation
- No PCA
- No Hyperparameter Tuning
- No Early Stopping for Overfitting
- Weight Feature Importance

# Seasonal Prediction ML Automation

- for every region and every season, train the same xgboost model, save all the results and save the model

- Data are seasonal, with each season labeled with its respective seasonal tercile

1993 MAM has 1993 MAM's tercile

In [132]:
# Ensure the output directories exist
os.makedirs(classification_report_folder_seasonal, exist_ok=True)
os.makedirs(confusion_matrix_folder_seasonal, exist_ok=True)
os.makedirs(feature_importance_folder_seasonal, exist_ok=True)
os.makedirs(forecast_details_folder_seasonal, exist_ok=True)
os.makedirs(model_folder_seasonal, exist_ok=True)
os.makedirs(shap_feature_importance_folder_seasonal, exist_ok=True)

# Loop over all CSVs in ML data folder
for filename in os.listdir(ML_data_folder_path):
    if filename.endswith('_seasonal.csv'):
        # Load the CSV
        df = pd.read_csv(os.path.join(ML_data_folder_path, filename))

        # Extract region name and season from file path
        region_name = '_'.join(filename.split('_')[0:-4])
        season = filename.split('_')[-4]

        # Status
        print(f"Currently Processing {region_name} Region, {season} Season")

        # Drop relevant columns based on the data if they are there
        if 'precip' in df.columns:
            df.drop(columns=['precip'], inplace=True)
        if 'region' in df.columns:
            df.drop(columns=['region'], inplace=True)
        if 'season' in df.columns:
            df.drop(columns=['season'], inplace=True)

        # Choose train and test years manually
        train_years = [1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003,
                       2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,
                       2014, 2015, 2016, 2017, 2018, 2019, 2020]

        test_years = [2021, 2022, 2023, 2024]

        # Manual split
        train_df = df[df['year'].isin(train_years)]
        test_df = df[df['year'].isin(test_years)]

        # Separate Features and labels
        X_train = train_df.drop(columns=['year', 'tercile'])
        X_test = test_df.drop(columns=['year', 'tercile'])
        y_train = train_df['tercile']
        y_test = test_df['tercile']

        # Encode labels (an: 1 0 0, bn: 0 0 1, n: 0 1 0)
        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_test_enc = le.transform(y_test)

        # Define model
        model = XGBClassifier(eval_metric='mlogloss')

        # Train the model on Training data and training labels
        model.fit(X_train, y_train_enc)

        # Predict classes
        y_pred = model.predict(X_test)

        # Predict probabilities for each class
        y_proba = model.predict_proba(X_test)

        # Save the model
        model_save_path = os.path.join(model_folder_seasonal, f"{region_name}_{season}_seasonal_model.json")
        model.save_model(model_save_path)

        # Save the classification report as a text file
        classification_report_path = os.path.join(classification_report_folder_seasonal, f"{region_name}_{season}_seasonal_model_classification_report.txt")
        with open(classification_report_path, 'w') as f:
            f.write(f"{region_name} {season} Long Lead Seasonal Model\n=== Classification Report ===\n")
            f.write(classification_report(
                y_test_enc,
                y_pred,
                labels=[0, 1, 2],  # Ensure all label indices are present
                target_names=le.classes_,
                zero_division=0  # Handle division by zero (i.e N was not in test data)
            ))

        # Confusion matrix
        conf_mat = confusion_matrix(y_test_enc, y_pred, labels=[0, 1, 2])
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_mat, annot=True, fmt='d',
                    xticklabels=le.classes_, yticklabels=le.classes_,
                    cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(f'{region_name} {season} \nLong Lead Seasonal Model\nConfusion Matrix')

        # Save the confusion matrix
        conf_matrix_file = os.path.join(confusion_matrix_folder_seasonal, f'{region_name}_{season}_seasonal_model_confusion_matrix.png')
        plt.savefig(conf_matrix_file, dpi=300)
        plt.close()

        # Feature importance plot
        plt.figure(figsize=(20, 12))
        plot_importance(model, max_num_features=20, importance_type='weight', height=0.5)
        plt.title(f'{region_name} {season} \nLong Lead Seasonal Model \nTop 20 Feature Importances')

        # Save the feature importance plot
        feature_importance_file = os.path.join(feature_importance_folder_seasonal, f'{region_name}_{season}_seasonal_model_feature_importance.png')
        plt.savefig(feature_importance_file, dpi=300)
        plt.close()

        # Create explainer
        explainer = shap.Explainer(model, X_train)

        # Compute SHAP values
        shap_values = explainer(X_test)

        # get the class names from the model's encoder
        class_names = le.classes_

        # Use matplotlib-compatible SHAP summary bar plot
        plt.figure(figsize=(20, 12))
        shap.summary_plot(shap_values.values, X_test, plot_type="bar", class_names=class_names, show=False) # use class names list to label shap plot
        plt.title(f'{region_name} {season} \nLong Lead Seasonal Model \nTop 20 SHAP Feature Importances')
        plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to fit title
        plt.savefig(os.path.join(shap_feature_importance_folder_seasonal, f'{region_name}_{season}_seasonal_model_shap_feature_importance.png'), dpi=300)
        plt.close()

        # Probabilistic forecast details

        # Reset index to ensure alignment
        test_info = test_df[['year']].reset_index(drop=True)  # Only 'year' for seasonal data
        proba_df = pd.DataFrame(y_proba, columns=[f"proba_{cls}" for cls in le.classes_])

        # Concatenate year and predicted probabilities
        results_df = pd.concat([test_info, proba_df], axis=1)

        # Add predicted and true class labels
        results_df['true_label'] = le.inverse_transform(y_test_enc)
        results_df['predicted_label'] = le.inverse_transform(y_pred)

        # Save the results
        forecast_details_file = os.path.join(forecast_details_folder_seasonal, f"{region_name}_{season}_seasonal_model_probabilistic_forecast_details.txt")
        with open(forecast_details_file, 'w') as f:
            f.write(f"{region_name} {season} Long Lead Seasonal Model\n=== Seasonal Predictions with Probabilities ===\n")
            f.write(results_df.to_string(index=False))


Currently Processing eastern_east_africa Region, OND Season
Currently Processing eastern_ukraine Region, JA Season
Currently Processing south_sudan Region, MJJ Season
Currently Processing eastern_east_africa Region, MAM Season
Currently Processing lake_victoria_basin Region, DJF Season
Currently Processing south_sudan Region, JAS Season
Currently Processing west_africa Region, JAS Season
Currently Processing lake_victoria_basin Region, SON Season
Currently Processing lake_victoria_basin Region, MAM Season
Currently Processing south_sudan Region, ASO Season
Currently Processing eastern_ukraine Region, DJF Season
Currently Processing sri_lanka Region, OND Season
Currently Processing southern_africa Region, DJF Season
Currently Processing southern_africa Region, FMA Season
Currently Processing eastern_ukraine Region, AMJ Season


<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

...

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

# Monthly Prediction ML Automation

- Data are monthly, with each month labeled with its respective monthly tercile

In [136]:
# Ensure the output directories exist
os.makedirs(confusion_matrix_folder_monthly, exist_ok=True)
os.makedirs(classification_report_folder_monthly, exist_ok=True)
os.makedirs(feature_importance_folder_monthly, exist_ok=True)
os.makedirs(forecast_details_folder_monthly, exist_ok=True)
os.makedirs(model_folder_monthly, exist_ok=True)
os.makedirs(shap_feature_importance_folder_monthly, exist_ok=True)

# Loop over all CSVs in ML data folder
for filename in os.listdir(ML_data_folder_path):
    if filename.endswith('_monthly.csv'):
        # Load the CSV
        df = pd.read_csv(os.path.join(ML_data_folder_path, filename))

        # Extract region name and season from file path
        region_name = '_'.join(filename.split('_')[0:-4])
        season = filename.split('_')[-4]

        # Status
        print(f"Currently Processing {region_name} Region, {season} Season")

        # Drop relevant columns based on the data if they are there
        if 'precip' in df.columns:
            df.drop(columns=['precip'], inplace=True)
        if 'region' in df.columns:
            df.drop(columns=['region'], inplace=True)
        if 'season' in df.columns:
            df.drop(columns=['season'], inplace=True)

        # Choose train and test years manually
        train_years = [1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003,
                       2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,
                       2014, 2015, 2016, 2017, 2018, 2019, 2020]

        test_years = [2021, 2022, 2023, 2024]

        # Manual split
        train_df = df[df['year'].isin(train_years)]
        test_df = df[df['year'].isin(test_years)]

        # Separate Features and labels
        X_train = train_df.drop(columns=['year', 'tercile'])
        X_test = test_df.drop(columns=['year', 'tercile'])
        y_train = train_df['tercile']
        y_test = test_df['tercile']

        # Encode labels (an: 1 0 0, bn: 0 0 1, n: 0 1 0)
        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_test_enc = le.transform(y_test)

        # Define model
        model = XGBClassifier(eval_metric='mlogloss')

        # Train the model on Training data and training labels
        model.fit(X_train, y_train_enc)

        # Predict classes
        y_pred = model.predict(X_test)

        # Predict probabilities for each class
        y_proba = model.predict_proba(X_test)

        # Save the model
        model_save_path = os.path.join(model_folder_monthly, f"{region_name}_{season}_monthly_model.json")
        model.save_model(model_save_path)

        # Save the classification report as a text file
        classification_report_path = os.path.join(classification_report_folder_monthly, f"{region_name}_{season}_monthly_model_classification_report.txt")
        with open(classification_report_path, 'w') as f:
            f.write(f"{region_name} {season} Long Lead Monthly Model\n=== Classification Report ===\n")
            f.write(classification_report(
                y_test_enc,
                y_pred,
                labels=[0, 1, 2],  # Ensure all label indices are present
                target_names=le.classes_,
                zero_division=0  # Handle division by zero (i.e N was not in test data)
            ))

        # Confusion matrix
        conf_mat = confusion_matrix(y_test_enc, y_pred, labels=[0, 1, 2])
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_mat, annot=True, fmt='d',
                    xticklabels=le.classes_, yticklabels=le.classes_,
                    cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(f'{region_name} {season} \nLong Lead Monthly Model\nConfusion Matrix')

        # Save the confusion matrix
        conf_matrix_file = os.path.join(confusion_matrix_folder_monthly, f'{region_name}_{season}_monthly_model_confusion_matrix.png')
        plt.savefig(conf_matrix_file, dpi=300)
        plt.close()

        # Feature importance plot
        plt.figure(figsize=(20, 12))
        plot_importance(model, max_num_features=20, importance_type='weight', height=0.5)
        plt.title(f'{region_name} {season} \nLong Lead Monthly Model \nTop 20 Feature Importances')

        # Save the feature importance plot
        feature_importance_file = os.path.join(feature_importance_folder_monthly, f'{region_name}_{season}_monthly_model_feature_importance.png')
        plt.savefig(feature_importance_file, dpi=300)
        plt.close()

        # Create explainer
        explainer = shap.Explainer(model, X_train)

        # Compute SHAP values
        shap_values = explainer(X_test)

        # get the class names from the model's encoder
        class_names = le.classes_

        # Use matplotlib-compatible SHAP summary bar plot
        plt.figure(figsize=(20, 12))
        shap.summary_plot(shap_values.values, X_test, plot_type="bar", class_names=class_names, show=False) # use class names list to label shap plot
        plt.title(f'{region_name} {season} \nLong Lead Monthly Model \nTop 20 SHAP Feature Importances')
        plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to fit title
        plt.savefig(os.path.join(shap_feature_importance_folder_monthly, f'{region_name}_{season}_monthly_model_shap_feature_importance.png'), dpi=300)
        plt.close()

        # Probabilistic forecast details

        # Reset index to ensure alignment
        test_info = test_df[['year', 'month']].reset_index(drop=True)  # 'year' and 'month' predictions
        proba_df = pd.DataFrame(y_proba, columns=[f"proba_{cls}" for cls in le.classes_])

        # Concatenate year and predicted probabilities
        results_df = pd.concat([test_info, proba_df], axis=1)

        # Add predicted and true class labels
        results_df['true_label'] = le.inverse_transform(y_test_enc)
        results_df['predicted_label'] = le.inverse_transform(y_pred)

        # Save the results
        forecast_details_file = os.path.join(forecast_details_folder_monthly, f"{region_name}_{season}_monthly_model_probabilistic_forecast_details.txt")
        with open(forecast_details_file, 'w') as f:
            f.write(f"{region_name} {season} Long Lead Monthly (Monthly Tercile) Model\n=== Monthly Predictions with Probabilities ===\n")
            f.write(results_df.to_string(index=False))


Currently Processing south_sudan Region, MJJ Season
Currently Processing west_africa Region, JAS Season
Currently Processing lake_victoria_basin Region, DJF Season
Currently Processing eastern_east_africa Region, MAM Season
Currently Processing eastern_ukraine Region, JA Season
Currently Processing lake_victoria_basin Region, SON Season
Currently Processing eastern_east_africa Region, OND Season
Currently Processing lake_victoria_basin Region, MAM Season
Currently Processing south_sudan Region, JAS Season
Currently Processing south_sudan Region, ASO Season
Currently Processing southern_africa Region, FMA Season
Currently Processing eastern_ukraine Region, AMJ Season
Currently Processing eastern_ukraine Region, DJF Season
Currently Processing sri_lanka Region, OND Season
Currently Processing southern_africa Region, DJF Season


<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

...

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

# Monthly (Seasonal Tercile) ML Automation
- Data are monthly, but each month is assigned its seasonal tercile category

- For example, 3, 4, 5 are all assigned AN if MAM was AN

In [139]:
# Ensure the output directories exist
os.makedirs(confusion_matrix_folder_monthly_seasonal, exist_ok=True)
os.makedirs(classification_report_folder_monthly_seasonal, exist_ok=True)
os.makedirs(feature_importance_folder_monthly_seasonal, exist_ok=True)
os.makedirs(forecast_details_folder_monthly_seasonal, exist_ok=True)
os.makedirs(model_folder_monthly_seasonal, exist_ok=True)
os.makedirs(shap_feature_importance_folder_monthly_seasonal, exist_ok=True)

# Loop over all CSVs in ML data folder
for filename in os.listdir(ML_data_folder_path):
    if filename.endswith('_monthly_seasonal_tercile.csv'):
        # Load the CSV
        df = pd.read_csv(os.path.join(ML_data_folder_path, filename))

        # Extract region name and season from file path
        region_name = '_'.join(filename.split('_')[0:-6])
        season = filename.split('_')[-6]

        # Status
        print(f"Currently Processing {region_name} Region, {season} Season")

        # Drop relevant columns based on the data if they are there
        if 'precip' in df.columns:
            df.drop(columns=['precip'], inplace=True)
        if 'region' in df.columns:
            df.drop(columns=['region'], inplace=True)
        if 'season' in df.columns:
            df.drop(columns=['season'], inplace=True)

        # Choose train and test years manually
        train_years = [1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003,
                       2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,
                       2014, 2015, 2016, 2017, 2018, 2019, 2020]

        test_years = [2021, 2022, 2023, 2024]

        # Manual split
        train_df = df[df['year'].isin(train_years)]
        test_df = df[df['year'].isin(test_years)]

        # Separate Features and labels
        X_train = train_df.drop(columns=['year', 'tercile'])
        X_test = test_df.drop(columns=['year', 'tercile'])
        y_train = train_df['tercile']
        y_test = test_df['tercile']

        # Encode labels (an: 1 0 0, bn: 0 0 1, n: 0 1 0)
        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_test_enc = le.transform(y_test)

        # Define model
        model = XGBClassifier(eval_metric='mlogloss')

        # Train the model on Training data and training labels
        model.fit(X_train, y_train_enc)

        # Predict classes
        y_pred = model.predict(X_test)

        # Predict probabilities for each class
        y_proba = model.predict_proba(X_test)

        # Save the model
        model_save_path = os.path.join(model_folder_monthly_seasonal, f"{region_name}_{season}_monthly_seasonal_tercile_model.json")
        model.save_model(model_save_path)

        # Save the classification report as a text file
        classification_report_path = os.path.join(classification_report_folder_monthly_seasonal, f"{region_name}_{season}_monthly_seasonal_tercile_model_classification_report.txt")
        with open(classification_report_path, 'w') as f:
            f.write(f"{region_name} {season} Long Lead Monthly (Seasonal Tercile) Model\n=== Classification Report ===\n")
            f.write(classification_report(
                y_test_enc,
                y_pred,
                labels=[0, 1, 2],  # Ensure all label indices are present
                target_names=le.classes_,
                zero_division=0  # Handle division by zero (i.e N was not in test data)
            ))

        # Confusion matrix
        conf_mat = confusion_matrix(y_test_enc, y_pred, labels=[0, 1, 2])
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_mat, annot=True, fmt='d',
                    xticklabels=le.classes_, yticklabels=le.classes_,
                    cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(f'{region_name} {season} \nLong Lead Monthly (Seasonal Tercile) Model\nConfusion Matrix')

        # Save the confusion matrix
        conf_matrix_file = os.path.join(confusion_matrix_folder_monthly_seasonal, f'{region_name}_{season}_monthly_seasonal_tercile_model_confusion_matrix.png')
        plt.savefig(conf_matrix_file, dpi=300)
        plt.close()

        # Feature importance plot
        plt.figure(figsize=(20, 12))
        plot_importance(model, max_num_features=20, importance_type='weight', height=0.5)
        plt.title(f'{region_name} {season} \nLong Lead Monthly Model (Seasonal Tercile) \nTop 20 Feature Importances')

        # Save the feature importance plot
        feature_importance_file = os.path.join(feature_importance_folder_monthly_seasonal, f'{region_name}_{season}_monthly_seasonal_tercile_model_feature_importance.png')
        plt.savefig(feature_importance_file, dpi=300)
        plt.close()

        # Create explainer
        explainer = shap.Explainer(model, X_train)

        # Compute SHAP values
        shap_values = explainer(X_test)

        # get the class names from the model's encoder
        class_names = le.classes_

        # Use matplotlib-compatible SHAP summary bar plot
        plt.figure(figsize=(20, 12))
        shap.summary_plot(shap_values.values, X_test, plot_type="bar", class_names=class_names, show=False) # use class names list to label shap plot
        plt.title(f'{region_name} {season} \nLong Lead Monthly (Seasonal Tercile) Model \nTop 20 SHAP Feature Importances')
        plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to fit title
        plt.savefig(os.path.join(shap_feature_importance_folder_monthly_seasonal, f'{region_name}_{season}_monthly_seasonal_tercile_model_shap_feature_importance.png'), dpi=300)
        plt.close()

        # Probabilistic forecast details

        # Reset index to ensure alignment
        test_info = test_df[['year', 'month']].reset_index(drop=True)  # 'year' and 'month' predictions
        proba_df = pd.DataFrame(y_proba, columns=[f"proba_{cls}" for cls in le.classes_])

        # Concatenate year and predicted probabilities
        results_df = pd.concat([test_info, proba_df], axis=1)

        # Add predicted and true class labels
        results_df['true_label'] = le.inverse_transform(y_test_enc)
        results_df['predicted_label'] = le.inverse_transform(y_pred)

        # Save the results
        forecast_details_file = os.path.join(forecast_details_folder_monthly_seasonal, f"{region_name}_{season}_monthly_model_seasonal_tercile_probabilistic_forecast_details.txt")
        with open(forecast_details_file, 'w') as f:
            f.write(f"{region_name} {season} Long Lead Monthly (Seasonal Tercile) Model\n=== Monthly Predictions with Probabilities ===\n")
            f.write(results_df.to_string(index=False))


Currently Processing eastern_east_africa Region, OND Season
Currently Processing lake_victoria_basin Region, MAM Season
Currently Processing eastern_east_africa Region, MAM Season
Currently Processing lake_victoria_basin Region, SON Season
Currently Processing lake_victoria_basin Region, DJF Season
Currently Processing west_africa Region, JAS Season
Currently Processing eastern_ukraine Region, JA Season
Currently Processing south_sudan Region, ASO Season
Currently Processing south_sudan Region, JAS Season
Currently Processing eastern_ukraine Region, AMJ Season
Currently Processing south_sudan Region, MJJ Season
Currently Processing eastern_ukraine Region, DJF Season
Currently Processing sri_lanka Region, OND Season
Currently Processing southern_africa Region, FMA Season
Currently Processing southern_africa Region, DJF Season


<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

...

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>

<Figure size 2000x1200 with 0 Axes>